In [1]:
import os
import xarray as xr
import numpy as np
import pandas as pd
import numpy.linalg as lin
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import Normalize
import cartopy.crs as ccrs
from matplotlib.colors import BoundaryNorm
from matplotlib.lines import Line2D
from matplotlib.font_manager import FontProperties
from xhistogram.xarray import histogram

In [2]:
plt.rcParams["font.family"] = "Arial"

In [7]:
# AMS standard figure widths in inches
ams_sizes = {
    "one_column": 3.2,
    "two_thirds": 4.5,
    "two_columns": 5.5,
    "more_than_two_columns": 6.5,
}

data1= xr.open_dataset('./../data/spatial/dist_data/raindist_data_100_1000km.nc') ## Reading in data. Current structure is that the high resolution distribution is available in both input files.
bincrates=data1.precipitation_bin[1:].values ## rain rate bins
otn=np.linspace(1,9,9)
xtickrates=np.append(0,otn*.1)
xtickrates=np.append(xtickrates,otn)
xtickrates=np.append(xtickrates,otn*10)
xtickrates=np.append(xtickrates,otn*100)
xticks=np.interp(xtickrates,bincrates,range(0,len(bincrates))); #% bin numbers associated with nice number rain rate
xticks,indices=np.unique(xticks,return_index=True)
xtickrates=xtickrates[indices]
### Bin width - needed to normalize the rain amount distribution
db=(bincrates[2]-bincrates[1])/bincrates[1];
# Common aspect ratio: width : height = 4:3 → height = width * 0.75
aspect_ratio = 1
width = ams_sizes["one_column"]
height = width * aspect_ratio

# Load rain amount distributions from the FROGS archive
froot = "./../data/spatial/frogs"
datasets = {
    "GPCP": "GPCP",
    "CMORPH_v1.0_CRT": "CMORPH",
    "GIRAFE": "GIRAFE",
    "PERSIANN_v1_r1": "PERSIANN",
}
resolutions = [200, 500, 1000]
colors = ['#41b6c4','#7fcdbb','#c7e9b4']
plabels = ['a)','b)','c)','d)']
fig, axes = plt.subplots(2, 2, figsize=(width * 2, height * 2 / 1.2), sharex=True)
axes = axes.flatten()

for ax, (dataset_key, dataset_label), plabel in zip(axes, datasets.items(), plabels):
    ymin, ymax = ax.get_ylim()
    t1 = ax.text(0.02, 0.95, plabel +' '+ dataset_label, transform=ax.transAxes,
                fontsize=10, fontweight='bold', va='top', ha='left')
    pamtH = data1.pamtH[1:].values
    ax.plot(np.arange(len(pamtH)), pamtH / db, color='#1d91c0', linewidth=1.5, label='1.0º')
    for res, color in zip(resolutions, colors):
        file_path = f"{froot}/raindist_{dataset_key}_100_{res}km.nc"
        if not os.path.exists(file_path):
            continue
        try:
            data = xr.open_dataset(file_path)
            bincrates = data.precipitation_bin[1:].values
            pamt = data.pamtL[1:].values
            db = (bincrates[2] - bincrates[1]) / bincrates[1]
            ax.plot(np.arange(len(pamt)), pamt / db, color=color, linewidth=1.5, label=str(res/100)+'º')
        except Exception as exc:
            print(f"Could not read {file_path}: {exc}")

    ax.set_xlim(4, 130)
    ax.set_ylim(0, 1.5)
    ax.set_ylabel("Rain amount (mm/d)", fontsize=10, fontweight="bold")
    for axis in ['top','bottom','left','right']:
        ax.spines[axis].set_linewidth(1.5)
    ax.tick_params(axis='both', labelsize=9, width=1.2)
    ax.set_xticks(xticks)
    ax.set_xticklabels(['0.1', '', '', '', '', '', '', '', '', '1', '', '', '', '', '', '', '', '', '10', '', '', '', '', '', '', '', '', '100', '', '', '', '', '', '', '', '', '1000'])
    ax.set_yticks(np.arange(0, 1.6, 0.25))
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
axes[2].set_xlabel("Rain rate bin", fontsize=10, fontweight="bold")
axes[3].set_xlabel("Rain rate bin", fontsize=10, fontweight="bold")
legend = axes[0].legend(fontsize=8, title_fontsize=9)
legend.get_title().set_fontweight('bold')
for text in legend.get_texts():
    text.set_fontweight('bold')

plt.tight_layout()
filename='./../figs/figS1_rain_amount.pdf'
plt.savefig(filename, format="pdf", bbox_inches="tight",dpi=300)
plt.close()

In [4]:
# AMS standard figure widths in inches
ams_sizes = {
    "one_column": 3.2,
    "two_thirds": 4.5,
    "two_columns": 5.5,
    "more_than_two_columns": 6.5,
}

# Common aspect ratio: width : height = 4:3 → height = width * 0.75
# You can change this based on the visual needs of your plot
aspect_ratio = 1


width = ams_sizes["two_columns"]  # Change this to the desired width from ams_sizes
height = width * aspect_ratio

data1= xr.open_dataset('./../data/spatial/dist_data/raindist_data_100_1000km.nc') ## Reading in data. Current structure is that the high resolution distribution is available in both input files.
bincrates=data1.precipitation_bin[1:].values ## rain rate bins
otn=np.linspace(1,9,9)
xtickrates=np.append(0,otn*.1)
xtickrates=np.append(xtickrates,otn)
xtickrates=np.append(xtickrates,otn*10)
xtickrates=np.append(xtickrates,otn*100)
xticks=np.interp(xtickrates,bincrates,range(0,len(bincrates))); #% bin numbers associated with nice number rain rate
xticks,indices=np.unique(xticks,return_index=True)
xtickrates=xtickrates[indices]
### Bin width - needed to normalize the rain amount distribution
db=(bincrates[2]-bincrates[1])/bincrates[1];
res=[10,25,50,100,200,500,1000]
colrs=['#081d58','#253494','#225ea8','#1d91c0','#41b6c4','#7fcdbb','#c7e9b4']
## Calculate rx1day by reading pre-computed rx1day files and averaging
res_labels = ["10km", "25km", "50km", "100km", "200km", "500km", "1000km"]

### Now we plot
plt.figure(figsize=(width,height), constrained_layout=True)
plt.clf()

ax=plt.subplot(111)
for i in range(0,len(res)):
    
    if i==0:
        data1= xr.open_dataset('./../data/spatial/dist_data/raindist_data_'+str(res[i])+'_25km.nc')## Reading in data. Current structure is that the high resolution distribution is available in both input files.
        ppdf = data1.ppdfH[1:].values ## rain amount of HR dataset
        dryd = 100*data1.ppdfH[0].values ## dry day frequency of HR dataset
    else:
        data1= xr.open_dataset('./../data/spatial/dist_data/raindist_data_10_'+str(res[i])+'km.nc')
        ppdf= data1.ppdfL[1:].values ## rain amount of LR dataset
        dryd = 100*data1.ppdfL[0].values ## dry day frequency of LR dataset
    plt.plot(range(0,len(ppdf)),100*ppdf/db, color=colrs[i], label=str(res[i]/100)+'º ('+"{:.1f}".format(dryd)+'%)',linewidth=1.5)
    plt.xlim((4,130))
    plt.ylim((0,20))
    legend_properties = {'weight':'bold','size':9}
    plt.legend(prop=legend_properties)

    ### Annotate with the dry day frequency
    ymin, ymax = ax.get_ylim()
    #t1=plt.text(115,ymax*0.90, 'a)',fontsize=12)
    #t2=plt.text(4,ymax*0.90, "{:.1f}".format(dryd)+'%',color='m')
    #plt.setp(t1,va='top',ha='left')
    #plt.setp(t2,va='top',ha='left')
    plt.setp(ax,xticks=xticks,xticklabels=['0.1','','','','','','','','','1','','','','','','','','','10','','','','','','','','','100','','','','','','','','','1000'])
    plt.xlabel('Rain rate (mm/d)',fontsize=12,fontweight="bold")
    plt.ylabel('Rain frequency (%)',fontsize=12,fontweight="bold")
    plt.xticks(fontsize=10,fontweight="bold")
    plt.yticks(fontsize=10,fontweight="bold")
    for axis in ['top','bottom','left','right']:
        ax.spines[axis].set_linewidth(1.5)

filename='./../figs/figS2_rain_frequency.pdf'
plt.savefig(filename, format="pdf", bbox_inches="tight",dpi=300)
plt.close()